# labone_helper functions inputs

In [ ]:
yaml_fn = generate_descriptor_yaml(
    filename = 'laboneq_helper/default_descriptor.yaml',      # optional
    devices = {                                               # takes "PQSC" , "HDAWG" , "SHFQC" , "SHFQA" , "SHFSG"
        "PQSC" : {'serial':'dev10000',                        # PQSC only requires these 2 arguments
                  'external_clock':True},                     # optional. Defaults to False
        "SHFQC" : [                                           # non-PQSC can have all of the below arguments
              {'serial':'dev20000', 
                  'usb':True,                                 # optional, Defaults to False
                  'external_clock':True,                      # optional. Defaults to False
                  'zsync':0,                                  # optional if no PQSC
                  'total_sg_channels': 1,                     # optional, sets to maximal value if unspecified (6 for QC, 8 for SG, 0 for QA and 8 for HDAWG)
                  'total_qa_channels': 1,                     # optional, sets to maximal value if unspecified (1 for QC, 0 for SG, 4 for QA and 0 for HDAWG)
                  'logical_signals_per_channel':2,            # optional, sets to maximal value if unspecified (8 for QC, 8 for SG, 0 for QA and 1 for HDAWG)
                  'ro_multiplex': 2                           # optional, sets to maximal value if unspecified (16 for QC, 0 for SG, 16 for QA and 0 for HDAWG)
              },
              {'serial':'dev20001',                           # add device of same type like this if multiple units are connected
                  'zsync':1},
            ],
        "HDAWG" : [                                           # add device of different type like this
              {'serial':'dev30001', 
                  'usb':True,                                 # optional, Defaults to False
                  'external_clock':True,                      # optional. Defaults to False
                  'zsync':2, 
                  'total_sg_channels': 2,}
              ],
        },
    n_qubits = 2                                              # optional. defaults to 1
    )


## 0.3 Function to generate calibration object.
Ideally, this should be generic enough to cover all needs.

from laboneq_helper.laboneq_helper import default_signal_map_and_calibration

see README file in laboneq_helper folder for details.

Currently, this should cover:

1. Fixed output frequency on any SG/QA channels
2. Utilizing up to 8 separate logical SG output lines, each with separate frequencies, per 1 physical SG channel
2. Frequency sweep through any SG/QA channels
3. Multiplexed QA measure/acquire (output/input) pair up to 16
5. Feedback based on QA input.
6. Output Router
7. Low Frequency mode (set LO to 0)
8. Muting function

Few things to note:

1. Modulation type of oscillator should be hardware in spectroscopy mode. Else, it should be set to software.
2. In using the feedback/discrimination function, threshold argument only accepts real values. Simple feedback/discrimination can be done by simply setting the threshold. SW oscillator in both measure/acquire should be set for feedback/discrimination use. (unclear if those can be used in spectroscopy mode)
3. However, for advanced feedback/discrimination, one may want to rotate the readout signal in the IQ plane to make them maximally separated in the I-axis. To do this, one needs to define an adequate integration kernel to rotate the readout. Such an integration kernel will have an oscillation term which behaves like the SW oscillator, hence the oscillator in the acquire line should be set to None. (it will output gibberish if you set both the SW oscillator and the correct kernel for discrimination) SW oscillator in the measure line on the other hand should be set as other typical measurements.

Next things to add:
1. Voltage offset
2. Amplitude settings for spectroscopy mode

In [ ]:
default_signal_map_and_calibration(
    sig_freq_map,                                           # see below for details
    metadata,                                               # see below for details
    qubit_index = "q0"                                      # optional. defaults to "q0". Format is f"q{index number}"
    )                                                       # Works for higher number of q's as long as it matches with the yaml descriptor generated

## Version before 2024/10/31

In [ ]:
sig_freq_map = {
    "dev12244":{        # provide the device serial number
        "SG0" :[        # Channels. Omittable if unused. For QC, "SG0"~"SG6" and "QA0". For SG, "SG0"~"SG8", For QA, "QA0"~"QA4", For HDAWG, "SG0"~"SG8"
            {"logical_signal" : 'drive_0_0', 
                "frequency" : qubit_parameters['q0']['freq_0_0'], 
                "range" : 0,                                                # optional. defaults to 0. SHFSG power range is in dBm, in multiples of 5, from -30 to 10. HDAWG power is from  0.2 ~ 5 in Volts
                "automute" : False,                                         # optional. defaults to False. Enables the automute for SGs. Unavailable for HDAWG
                "route" : [                                                 # optional. This is how output routers are assigned. Up to 3 connections can be made. Unavailable for HDAWG
                    {"src": 'drive_1_0', "amp_scaling": 1,"ph_shift": 0},   # Routings are made to physical channels. 
                    {"src": 'drive_2_0', "amp_scaling": 1,"ph_shift": 0.}   # A single logical signal from a physical channel is enough to enable the router.
                    ]                                                       # Trying to connect multiple logical signal from a single physical channel will prompt compiler error
                },
        ],
        "SG1" :[
            {"logical_signal" : 'drive_1_0',                                # This is how multiple logical signals are assigned to a single physical channel.
                "frequency" : qubit_parameters['q0']['freq_1_0'],           # Up to 8 logical channels per physical channel are supported.
                },
            {"logical_signal" : 'drive_1_1', 
                "frequency" : qubit_parameters['q0']['freq_1_1'], 
                },
        ],
        # "SG2" :[], "SG3":[], "SG4":[], "SG5":[],
        "QA0":[
                {"logical_signal":["measure_0", "acquire_0"],   
                 "frequency":qubit_parameters['q0']['ro_freq'], 
                 "range":[0,0],                                 # optional. defaults to [0,0]. SHFQA [output,input] power range is in dBm, in multiples of 5, from [-30 to 10, -50 to 10]
                 "automute" : False,                            # optional. defaults to False. Enables the automute for QA out
                 "spectroscopy": False,                         # optional. defaults to False. 
                 "rotate_ro": False,                            # optional. defaults to False. Set to True if you want to rotate the readout in the IQ plane through acquire kernel.
                 "thresholds": None,                            # optional. defaults to None. Take a list[], and only real values
                 },
                {"logical_signal":["measure_1", "acquire_1"],   # List of logical signals for measure/acquire. Multiplexed readout up to 16 signals
                 "frequency":qubit_parameters['q0']['ro_freq_1'], 
                },
            ],
    },
}

## Version after 2024/10/31
Using dictionaries instead of lists for logitcal signals for clarity

In [ ]:
sig_freq_map = {
    "dev12244":{        # provide the device serial number
        "SG0" :{        # Channels. Omittable if unused. For QC, "SG0"~"SG6" and "QA0". For SG, "SG0"~"SG8", For QA, "QA0"~"QA4", For HDAWG, "SG0"~"SG8"
            'drive_0_0':{ 
                "frequency" : qubit_parameters['q0']['freq_0_0'], 
                "range" : 0,                                                # optional. defaults to 0. SHFSG power range is in dBm, in multiples of 5, from -30 to 10. HDAWG power is from  0.2 ~ 5 in Volts
                "automute" : False,                                         # optional. defaults to False. Enables the automute for SGs. Unavailable for HDAWG
                "route" : [                                                 # optional. This is how output routers are assigned. Up to 3 connections can be made. Unavailable for HDAWG
                    {"src": 'drive_1_0', "amp_scaling": 1,"ph_shift": 0},   # Routings are made to physical channels. 
                    {"src": 'drive_2_0', "amp_scaling": 1,"ph_shift": 0.}   # A single logical signal from a physical channel is enough to enable the router.
                    ]                                                       # Trying to connect multiple logical signal from a single physical channel will prompt compiler error
                },
        },
        "SG1" :{
            'drive_1_0':{                                # This is how multiple logical signals are assigned to a single physical channel.
                "frequency" : qubit_parameters['q0']['freq_1_0'],           # Up to 8 logical channels per physical channel are supported.
                },
            'drive_1_1':{ 
                "frequency" : qubit_parameters['q0']['freq_1_1'], 
                },
        },
        # "SG2" :[], "SG3":[], "SG4":[], "SG5":[],
        "QA0":{
            "measure_0/acquire_0" : {                       # key of dictionary has to be meausure/acquire pair delimited by "/"
                "frequency":qubit_parameters['q0']['ro_freq'], 
                "range":[0,0],                                 # optional. defaults to [0,0]. SHFQA [output,input] power range is in dBm, in multiples of 5, from [-30 to 10, -50 to 10]
                "automute" : False,                            # optional. defaults to False. Enables the automute for QA out
                "spectroscopy": False,                         # optional. defaults to False. 
                "rotate_ro": False,                            # optional. defaults to False. Set to True if you want to rotate the readout in the IQ plane through acquire kernel.
                "thresholds": None,                            # optional. defaults to None. Take a list[], and only real values
                },
            "measure_1/acquire_1" : {   # List of logical signals for measure/acquire. Multiplexed readout up to 16 signals
                "frequency":qubit_parameters['q0']['ro_freq_1'], 
                },
        },
    },
}

In [ ]:
metadata = {"device_setup" : device_setup,                  # how metadata is defined. 
            "lo_settings": lo_settings,                     # device_setup, lo_settings, and qubit parameters are global variables in the experiment file
            "qubit_parameters": qubit_parameters}

In [ ]:
# How lo_settings is defined

def device_lo_settings():
    return {                    # list all devices used, except for the PQSC
        serial_num: {           # example is with QC. uses "QA0_LO", "SG0_LO", "SG2_LO", "SG4_LO"
            # SHFQA LO Frequency
            "QA0_LO": 0,        # with SHFQA, "QA0_LO", "QA1_LO", "QA2_LO", "QA3_LO"
            # SHFSG LO Frequencies, one center frequency per two channels on SHFQC
            "SG0_LO": 0,        # with SHFSG, "SG0_LO", "SG2_LO", "SG4_LO", "SG6_LO"
            "SG2_LO": 0,
            "SG4_LO": 0,
        },
        # 'dev8433': { # For HDAWG, all LOs should be 0 or be omitted since HDAWG does not have an LO.
        #     "SG0_LO": 0,
        #     "SG2_LO": 0,
        #     "SG4_LO": 0,
        #     "SG6_LO": 0
        # }
    }

lo_settings = {
    k: device_lo_settings() for k in device_setup.logical_signal_groups.keys()
}

In [2]:
map

map